# 06 Narrow and Wide Transformations

## Goal

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 Goal</b><br>
Tell narrow from wide transformations by their <b>partition dependencies</b>, predict which operations shuffle, and use that knowledge to reduce shuffles: co-partitioning, <code>coalesce</code> vs <code>repartition</code>, broadcast joins and pipelining.
</div>

## Why it matters

<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Why it matters</b><br>
Wide transformations cause shuffles, and shuffles are the main cost in most Spark jobs. Every stage boundary, every "spill to disk" warning and most slow joins come back to one wide transformation. The narrow/wide distinction also explains fault tolerance: how much work Spark redoes when a task fails.
</div>

## Concept

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 Dependencies between partitions</b>
<ul><li><b>Narrow dependency</b>: each output partition is computed from <b>one</b> input partition (or a fixed few, as with <code>coalesce</code> or <code>union</code>). Rows don't move between executors. Narrow steps are <b>pipelined</b> into a single stage</li><li><b>Wide dependency</b>: an output partition needs data from <b>many</b> (often all) input partitions. Rows with the same key must be moved together, which is a <b>shuffle</b>, and that ends the stage</li></ul>
</div>

```
NARROW (filter, select, withColumn)          WIDE (groupBy, join, distinct, orderBy)

 in P1 ──▶ out P1                             in P1 ──┬──▶ out P1 (keys A-F)
 in P2 ──▶ out P2                             in P2 ──┼──▶ out P2 (keys G-M)
 in P3 ──▶ out P3                             in P3 ──┴──▶ out P3 (keys N-Z)
 (no data movement)                           (every input sends to every output: shuffle)
```

## Real-world example

<div style="background:#FFF4DB;border-left:5px solid #E5A100;padding:10px 14px;border-radius:6px;color:#4a3500;">
<b>🏦 Real-world example</b><br>
A job did <code>repartition(200)</code> after reading, then <code>groupBy("store")</code>, then <code>orderBy("store")</code> before writing. That's three shuffles. The <code>repartition</code> was unnecessary (the <code>groupBy</code> shuffles anyway), and the <code>orderBy</code> wasn't needed by the consumers. Removing both cut the runtime in half without changing the output data.
</div>

## Classification

| Narrow | Wide |
|---|---|
| `select`, `withColumn`, `drop`, `filter` | `groupBy().agg()`, `distinct`, `dropDuplicates` |
| `union`, `unionByName` | `join` (sort-merge / shuffled hash) |
| `coalesce(n)` (merging partitions) | `repartition(n)`, `repartition(col)` |
| `explode`, `sample`, `limit` (per partition) | `orderBy` / `sort` (global) |
| `sortWithinPartitions` | Window functions with `partitionBy` |
| Join with a **broadcast** small side (big side stays narrow) | `intersect`, `subtract` |

## Setup

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Defines a helper that counts shuffle exchanges in a plan, and a sample DataFrame.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
No output.
</div>

In [0]:
import io, contextlib, re
from pyspark.sql import functions as F, Window

def plan_text(df):
    buf = io.StringIO()
    with contextlib.redirect_stdout(buf):
        df.explain()
    return buf.getvalue()

def shuffles(df):
    """Number of shuffle exchanges (broadcasts are not shuffles)."""
    return len(re.findall(r"(?<!Broadcast)Exchange (hash|range|RoundRobin|SinglePartition)", plan_text(df)))

sales = (
    spark.range(0, 1_000_000, 1, numPartitions=8)
    .withColumns({"store": F.col("id") % 50, "amount": ((F.col("id") % 997) / 10).cast("decimal(10,1)")})
)

## 1. Classify by evidence

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Runs a list of operations through the <code>shuffles</code> helper.<br><br>
<b>Why it matters</b><br>Memorizing the table is useful, but checking the plan is definitive. This is the habit to build: when in doubt, count the exchanges.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
0 shuffles for the narrow operations and 1 for each wide one. The window function and <code>orderBy</code> each add one.
</div>

In [0]:
operations = {
    "select": sales.select("store", "amount"),
    "filter": sales.filter("amount > 50"),
    "withColumn": sales.withColumn("tax", F.col("amount") * 0.2),
    "union": sales.union(sales),
    "coalesce(2)": sales.coalesce(2),
    "explode": sales.withColumn("x", F.explode(F.array(F.lit(1), F.lit(2)))),
    "sortWithinPartitions": sales.sortWithinPartitions("amount"),
    "groupBy": sales.groupBy("store").count(),
    "distinct": sales.select("store").distinct(),
    "orderBy": sales.orderBy("amount"),
    "repartition(16)": sales.repartition(16),
    "window": sales.withColumn("rn", F.row_number().over(Window.partitionBy("store").orderBy("amount"))),
}
for name, df in operations.items():
    n = shuffles(df)
    print(f"{name:<22} shuffles: {n}  -> {'wide' if n else 'narrow'}")

## 2. coalesce vs repartition

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Reduces 8 partitions to 2 with <code>coalesce</code> and with <code>repartition</code>, and compares plans and partition sizes.<br><br>
<b>Why it matters</b><br><code>coalesce</code> is narrow: it merges neighbouring partitions without a shuffle, which is cheap but can leave uneven partitions. <code>repartition</code> is wide: it shuffles every row to produce evenly sized partitions. Use <code>coalesce</code> to reduce partitions cheaply (for example before writing), and <code>repartition</code> to increase them or rebalance.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
<code>coalesce(2)</code>: 0 shuffles, 2 partitions. <code>repartition(2)</code>: 1 shuffle (<code>RoundRobinPartitioning</code>), 2 balanced partitions.
</div>

In [0]:
def sizes(df):
    return [r["count"] for r in df.groupBy(F.spark_partition_id().alias("p")).count().orderBy("p").collect()]

print("coalesce(2)    shuffles:", shuffles(sales.coalesce(2)), "sizes:", sizes(sales.coalesce(2)))
print("repartition(2) shuffles:", shuffles(sales.repartition(2)), "sizes:", sizes(sales.repartition(2)))

## 3. Narrow steps are pipelined into one stage

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Chains several narrow transformations and shows they share one code-generated stage (<code>*(1)</code>).<br><br>
<b>Why it matters</b><br>Spark doesn't materialize anything between narrow steps: each row flows through all of them in one loop. Ten narrow steps cost about the same as one pass over the data. Wide steps are what create extra passes.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
A physical plan where <code>Range</code>, <code>Project</code> and <code>Filter</code> all carry the same <code>*(1)</code> prefix, and no <code>Exchange</code>.
</div>

In [0]:
(
    sales.withColumn("tax", F.col("amount") * 0.2)
    .filter("tax > 5")
    .withColumn("bucket", F.col("store") % 5)
    .select("store", "bucket", "tax")
).explain()

## 4. Avoid a shuffle with a broadcast join

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Joins sales to a small store table, first as a sort-merge join and then as a broadcast join, and counts the shuffles.<br><br>
<b>Why it matters</b><br>A normal join is wide on <b>both</b> sides. Broadcasting the small side makes the join narrow for the big side: each executor joins its own partitions against a local copy of the small table.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
Sort-merge join: 2 shuffles (both sides). Broadcast join: 0 shuffles.
</div>

In [0]:
stores = spark.range(50).withColumnRenamed("id", "store").withColumn("region", F.col("store") % 3)

print("sort-merge join shuffles:", shuffles(sales.join(stores.hint("merge"), "store")))
print("broadcast join shuffles :", shuffles(sales.join(F.broadcast(stores), "store")))

## 5. Reuse an existing partitioning

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Repartitions data by <code>store</code> once, then runs a <code>groupBy("store")</code> and a window by <code>store</code>, and shows that Spark doesn't shuffle again.<br><br>
<b>Why it matters</b><br>If data is already hash-partitioned by the key, later wide operations on the <b>same key</b> can reuse that layout. Spark tracks the output partitioning and skips redundant exchanges. Repartitioning by the key you'll aggregate on is only worth it when several wide operations share it.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
1 shuffle in total for <code>repartition("store")</code> followed by <code>groupBy("store")</code> and a window partitioned by <code>store</code>. Grouping by a different column adds a second shuffle.
</div>

In [0]:
by_store = sales.repartition("store")

same_key = by_store.groupBy("store").agg(F.sum("amount").alias("total")) \
                   .withColumn("rank", F.rank().over(Window.partitionBy("store").orderBy("total")))
other_key = by_store.groupBy((F.col("id") % 7).alias("other")).count()

print("same key shuffles :", shuffles(same_key))
print("other key shuffles:", shuffles(other_key))

## 6. Fault tolerance: lineage and shuffle files

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Explains what Spark recomputes when a task fails, for narrow and wide dependencies.<br><br>
<b>Why it matters</b>
<ul><li>Narrow: a lost partition is recomputed from its <b>one</b> parent partition. Cheap</li><li>Wide: a lost post-shuffle partition needs data from <b>all</b> parent partitions. Spark avoids recomputing them by keeping the <b>shuffle files</b> written by the map side. If an executor holding shuffle files dies, the map tasks that wrote them must be re-run</li></ul>
This is why long chains of wide transformations are more fragile and more expensive to recover.
</div>

```
filter (narrow):    lost out-P2  <──  recompute in-P2 only
groupBy (wide):     lost out-P2  <──  re-read shuffle blocks from ALL map outputs
                                      (re-run map tasks whose shuffle files were lost)
```

## Under the hood

```
narrow chain:  Scan → Filter → Project → Project       one stage, one loop per partition
wide step:     ... → HashAggregate(partial)
                      Exchange hashpartitioning(key)    ← map side writes shuffle files
                     HashAggregate(final) → ...         ← reduce side reads them
```

<span style="background:#1E9E5A;color:white;padding:2px 10px;border-radius:12px;font-size:12px;">Narrow</span> pipelined, no data movement, cheap recovery.

<span style="background:#D64545;color:white;padding:2px 10px;border-radius:12px;font-size:12px;">Wide</span> shuffle, a new stage, expensive in disk, network and memory.

**Interview soundbite:** "Narrow transformations are pipelined inside a stage. Wide transformations need data with the same key in the same partition, so they shuffle and create a stage boundary. Optimizing Spark is largely about removing unnecessary wide transformations and making the necessary ones move less data."

In [0]:
sales.filter("amount > 50").groupBy("store").agg(F.sum("amount")).explain()

## Common mistakes and troubleshooting log

<div style="background:#FDEAEA;border-left:5px solid #D64545;padding:10px 14px;border-radius:6px;color:#4a1414;">
<b>⛔ Problem: <code>repartition</code> right before a <code>groupBy</code> on another key</b><br>
Two shuffles where one would do. Remove the repartition, or repartition by the grouping key only if several operations reuse it.<br><br>
**⛔ Problem: <code>coalesce(1)</code> makes the whole job slow**<br><code>coalesce</code> is narrow, so it can pull upstream work into a single task too. Use <code>repartition(1)</code> if the upstream work should stay parallel, or avoid a single output file.<br><br>
**⛔ Problem: an unnecessary global <code>orderBy</code>**<br>Sort only if the consumer needs the order. <code>sortWithinPartitions</code> is narrow.<br><br>
<b>⛔ Problem: joining a small lookup table shuffles a huge fact table</b><br>Broadcast the small side.<br><br>
<b>✍️ My own problems</b><br>(add yours here)
</div>

## Interview Q&A

<div style="background:#F1E9FF;border-left:5px solid #7A4DD8;padding:10px 14px;border-radius:6px;color:#2d1a57;">
<b>🎤 1. What is the difference between narrow and wide transformations?</b><br>
In a narrow transformation each output partition depends on one input partition, so there's no data movement and steps are pipelined in one stage. In a wide transformation an output partition depends on many input partitions, so data must be shuffled across the cluster and a new stage starts.<br><br>

<b>🎤 2. Give examples of each.</b><br>
Narrow: <code>select</code>, <code>filter</code>, <code>withColumn</code>, <code>union</code>, <code>coalesce</code>, <code>explode</code>. Wide: <code>groupBy</code>, <code>join</code> (non-broadcast), <code>distinct</code>, <code>orderBy</code>, <code>repartition</code>, window functions with <code>partitionBy</code>.<br><br>

<b>🎤 3. Is <code>coalesce</code> narrow or wide? And <code>repartition</code>?</b><br>
<code>coalesce</code> is narrow, because it merges existing partitions without a shuffle. <code>repartition</code> is wide, because it shuffles every row to create new, balanced partitions.<br><br>

<b>🎤 4. How can a join avoid a shuffle?</b><br>
By broadcasting the small side, so the big side is joined partition by partition without moving. Or if both sides are already partitioned by the join key, for example after a repartition or with bucketing on classic Hive tables.<br><br>

<b>🎤 5. Why do wide transformations affect fault tolerance?</b><br>
A lost partition after a shuffle depends on all parent partitions. Spark keeps shuffle files so it doesn't recompute everything, but if those files are lost with an executor, the map tasks must be re-run.
</div>

## Certification corner

<div style="background:#E6F6F7;border-left:5px solid #0E8C8C;padding:10px 14px;border-radius:6px;color:#0b3536;">
<b>🎓 Exam-style questions (Data Engineer Associate)</b><br><br>
<b>Q1. Which pair contains only narrow transformations?</b><br>
A. <code>filter</code> and <code>groupBy</code><br>
B. <code>select</code> and <code>withColumn</code><br>
C. <code>join</code> and <code>distinct</code><br>
D. <code>orderBy</code> and <code>repartition</code><br>
<details><summary><b>Show answer</b></summary><b>B.</b></details><br><br>

<b>Q2. A data engineer needs to reduce a DataFrame from 400 partitions to 10 before writing, as cheaply as possible. Which method should they use?</b><br>
A. <code>repartition(10)</code><br>
B. <code>coalesce(10)</code><br>
C. <code>orderBy</code> then <code>limit</code><br>
D. <code>distinct()</code><br>
<details><summary><b>Show answer</b></summary><b>B.</b> <code>coalesce</code> reduces partitions without a full shuffle.</details>
</div>

## Practice problem

<div style="background:#FFEFE5;border-left:5px solid #F07D2E;padding:10px 14px;border-radius:6px;color:#4a2300;">
<b>🧪 Without copying from above:</b>
<ol><li>Build a query with exactly 3 shuffles, then rewrite it to produce the same result with 1 shuffle</li><li>Show the partition sizes after <code>coalesce(3)</code> and after <code>repartition(3)</code> of <code>sales</code></li><li>Join <code>sales</code> to <code>stores</code> and compute total amount per <code>region</code> with only 1 shuffle</li></ol>
</div>

In [0]:
# Your solution for the practice problem goes here.
# Add a comment above each step explaining what it does.

## Solution

<div style="background:#FFF4DB;border-left:5px solid #E5A100;padding:10px 14px;border-radius:6px;color:#4a3500;">
<b>⚠️ Try it yourself first</b><br>
Only look at this after you have a working answer of your own.
</div>

In [0]:
# 1. Three shuffles: repartition + groupBy + orderBy. One shuffle: just the groupBy.
three = sales.repartition(20).groupBy("store").agg(F.sum("amount").alias("t")).orderBy("store")
one = sales.groupBy("store").agg(F.sum("amount").alias("t"))
print("three:", shuffles(three), "| one:", shuffles(one))
print("same data:", sorted(three.collect()) == sorted(one.collect()))

# 2. Partition sizes
print("coalesce(3):", sizes(sales.coalesce(3)))
print("repartition(3):", sizes(sales.repartition(3)))

# 3. Broadcast the small side so only the aggregation shuffles
per_region = sales.join(F.broadcast(stores), "store").groupBy("region").agg(F.sum("amount").alias("total"))
print("shuffles:", shuffles(per_region))
per_region.orderBy("region").show()

## Summary and cheat sheet

<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Key takeaways</b>
<ul><li>Narrow: one input partition per output partition, no shuffle, pipelined into one stage</li><li>Wide: many input partitions per output partition, shuffle, new stage</li><li><code>coalesce</code> (narrow, cheap, possibly uneven) vs <code>repartition</code> (wide, balanced)</li><li>Broadcast joins keep the big side narrow</li><li>Data already partitioned by a key can be reused by later wide operations on the same key</li><li>Recovery: narrow recomputes one parent, wide relies on shuffle files from all parents</li><li>Count <code>Exchange</code> nodes in <code>explain()</code> to know for sure</li></ul>
</div>

| Want | Use |
|---|---|
| Fewer partitions, cheaply | `coalesce(n)` |
| More or balanced partitions | `repartition(n)` |
| Co-locate keys for several operations | `repartition("key")` |
| Join a small table without shuffling the big one | `F.broadcast(small)` |
| Sort without a shuffle | `sortWithinPartitions` |

## Git commit

```
git add 02_spark_internals/06_narrow_wide_transformations.ipynb
git commit -m "add 02_06 narrow wide transformations notebook"
```